### Tools

Models can request to call tools that perform task such as fetching data from the database, searching the web, or runnning code. Tools are pairing of:

1. A Schema, including the name of the tool, description, and/or argument defination(often a JSON format)
2. A function or coroutine to execute

In [ ]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] =  os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")

response= model.invoke("Write a story in 50 words")

ValueError: Unable to infer model provider for model='qwen/qwen3-32b'. Please specify 'model_provider' directly.

Supported providers: anthropic, anthropic_bedrock, azure_ai, azure_openai, baseten, bedrock, bedrock_converse, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggingface, ibm, litellm, mistralai, nvidia, ollama, openai, openrouter, perplexity, together, upstage, xai

For help with specific providers, see: https://docs.langchain.com/oss/python/integrations/providers

In [ ]:
response

AIMessage(content='As the sun set over the rolling hills, a lone violinist stood on a small stage, her music dancing in the wind. The crowd was mesmerized by her soulful melodies, their faces aglow with emotion. In that moment, all worries faded, and the beauty of music reigned supreme.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 42, 'total_tokens': 104, 'completion_time': 0.108730785, 'completion_tokens_details': None, 'prompt_time': 0.002348724, 'prompt_tokens_details': None, 'queue_time': 0.045619346, 'total_time': 0.111079509}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d5f5b-1e85-73b1-a819-2ec0fbddb5aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 62, 'total_tokens': 104})

In [ ]:
print("Hello")

Hello


In [ ]:
## Tools
from langchain.tools import tool

@tool # decorator
def get_weather(location: str) -> str:
    """Get the weather at any location """
    return f"The current weather is sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [ ]:

response = model_with_tools.invoke("What's the weather in Mumbai?")
print(response)
# response.tool_calls

content='' additional_kwargs={'tool_calls': [{'id': 'ezh3m6zhj', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.02892216, 'completion_tokens_details': None, 'prompt_time': 0.024755931, 'prompt_tokens_details': None, 'queue_time': 0.098981447, 'total_time': 0.053678091}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d5f5b-2475-7012-b26c-9272ec22b872-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'ezh3m6zhj', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}


### Tool execution loop

In [ ]:
# step 1: Model generates tools call
messages  = [{"role": "user", "content": "What's the wearher in Mumbai?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
# messages

# step 2: Execution tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to the LLM model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

However, please note that the actual weather in Mumbai is obtained by calling the 'get_weather' function which is currently unavailable in this environment.


In [ ]:
messages

[{'role': 'user', 'content': "What's the wearher in Mumbai?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '75pgpf5rd', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.02063387, 'completion_tokens_details': None, 'prompt_time': 0.017950133, 'prompt_tokens_details': None, 'queue_time': 0.045779286, 'total_time': 0.038584003}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d5f5b-2580-7d73-9bdf-be5f22a5fca0-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': '75pgpf5rd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}),
 ToolMessage(content='The curr